# LRS-Net Training

Lightweight Remote-Sensing Super-Resolution Network — GPU training notebook for Colab/Kaggle.

Run top to bottom. Everything that needs a GPU happens here; model comparison, the params/PSNR Pareto plot, and the downstream-task evaluation happen back on CPU locally using the weights, `history.csv`, and figures this notebook writes out.

**Expected data layout** — flat, no pre-split train/val/test subfolders (the split happens in code, see §3-4):
```
DATA_DIR/
  HR/*.png
  LR/*.png   # omit entirely (LR_DIR = None) to use synthetic bicubic degradation instead
```
For real (non-synthetic) LR/HR pairs instead of the RSA data, see §2b to convert the WorldStrat dataset into this same layout.

## 0. Check GPU

In [ ]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

## 1. Get the code
Clones the repo so `lrsnet/` is importable. Re-run this cell after pushing local changes to pull the latest version.

In [ ]:
import os

IN_KAGGLE = os.path.exists("/kaggle/working")
WORKDIR = "/kaggle/working" if IN_KAGGLE else "/content"

REPO_URL = "https://github.com/Wiredu2020/Super-Resolution-Methods-For-Satellite-Image-Applications.git"
REPO_DIR = os.path.join(WORKDIR, "Super-Resolution-Methods-For-Satellite-Image-Applications")

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

import sys
sys.path.append(os.path.join(REPO_DIR, "LRS-Net"))

## 2. Data & output paths
On Kaggle: add your dataset via "+ Add Input" in the sidebar first, and turn on internet access in Settings. On Colab: this mounts Drive.

In [ ]:
if IN_KAGGLE:
    DATA_DIR = "/kaggle/input/<your-dataset-slug>"   # replace with the dataset you added
    OUTPUT_DIR = "/kaggle/working/outputs/LRSNet"
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/SatelliteSR/data"        # adjust to your Drive path
    OUTPUT_DIR = "/content/drive/MyDrive/SatelliteSR/outputs/LRSNet"

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2b. Optional: convert WorldStrat to the flat HR/LR PNG layout

Skip this if training on the RSA data (§3 points straight at `Processed_Data`).

Real (non-synthetic) LR/HR pairs from [WorldStrat](https://arxiv.org/abs/2207.06418) — Airbus SPOT 6/7 HR (1.5 m) temporally matched with Sentinel-2 LR (10 m). Easiest source on Kaggle: add the [`jucor1/worldstrat`](https://www.kaggle.com/datasets/jucor1/worldstrat) dataset as input, no download script needed. (License: HR Airbus imagery is CC-BY-NC 4.0 — fine for thesis use.)

The on-disk layout differs between releases, so **run `inspect()` first and confirm the globs below actually match** before exporting — don't skip straight to export_to_png on a guess.

In [ ]:
from lrsnet import worldstrat

WORLDSTRAT_ROOT = "/kaggle/input/worldstrat" if IN_KAGGLE else "/content/drive/MyDrive/worldstrat"

worldstrat.inspect(WORLDSTRAT_ROOT)

In [ ]:
# Adjust hr_glob/lr_glob to match what inspect() printed above, then run.
# hr_band_indices/lr_band_indices: 1-based rasterio band numbers -- verify
# against the dataset's band-order docs before trusting the defaults.
hr_paths, lr_paths = worldstrat.find_pairs(
    WORLDSTRAT_ROOT, hr_glob="*/hr.tif", lr_glob="lr_revisit_0.tif"
)
print(f"Found {len(hr_paths)} pairs")

# Input datasets are read-only on Kaggle -- write converted PNGs to OUTPUT_DIR instead.
WORLDSTRAT_HR_DIR = os.path.join(OUTPUT_DIR, "data/worldstrat_HR")
WORLDSTRAT_LR_DIR = os.path.join(OUTPUT_DIR, "data/worldstrat_LR")

worldstrat.export_to_png(
    list(zip(hr_paths, lr_paths)), WORLDSTRAT_HR_DIR, WORLDSTRAT_LR_DIR
)

# Point §3's HR_DIR/LR_DIR at these after running this cell:
#   HR_DIR, LR_DIR = WORLDSTRAT_HR_DIR, WORLDSTRAT_LR_DIR

## 3. Config

In [ ]:
SCALE = 2
HR_SIZE = 512
BATCH_SIZE = 8
EPOCHS = 30
NUM_FILTERS = 32
NUM_BLOCKS = 6

VAL_FRACTION = 0.1
TEST_FRACTION = 0.1
SEED = 42  # fixes the train/val/test split so it's reproducible across runs/models

# RSA data (Processed_Data): flat, matched-by-filename HR/ and LR/ folders.
HR_DIR = os.path.join(DATA_DIR, "HR")
LR_DIR = os.path.join(DATA_DIR, "LR")
# Set LR_DIR = None to fall back to synthetic bicubic degradation instead of real pairs.
# For WorldStrat instead, run §2b then uncomment:
# HR_DIR, LR_DIR = WORLDSTRAT_HR_DIR, WORLDSTRAT_LR_DIR

## 4. Datasets

In [ ]:
from lrsnet import data, losses, metrics, utils, build_lrsnet

datasets = data.load_split_datasets(
    HR_DIR, LR_DIR,
    val_fraction=VAL_FRACTION, test_fraction=TEST_FRACTION, seed=SEED,
    save_split_json=os.path.join(OUTPUT_DIR, "split.json"),  # keep this: reused for the CPU-side test-set comparison
    scale=SCALE, hr_size=HR_SIZE, batch_size=BATCH_SIZE,
)
train_ds, val_ds, test_ds = datasets["train"], datasets["val"], datasets["test"]

## 5. Build LRS-Net

In [ ]:
model = build_lrsnet(num_filters=NUM_FILTERS, num_blocks=NUM_BLOCKS, scale=SCALE)
model.summary()
print(metrics.count_params(model))

## 6. Compile

In [ ]:
lr_schedule = tf.keras.optimizers.schedules.PiecewiseConstantDecay(
    boundaries=[5000], values=[1e-4, 5e-5]
)
optimizer = tf.keras.optimizers.Adam(learning_rate=lr_schedule)

model.compile(
    optimizer=optimizer,
    loss=losses.l1_ssim_loss(),
    metrics=[metrics.PSNR, metrics.ssim_metric],
)

## 7. Callbacks

In [ ]:
weights_dir = os.path.join(OUTPUT_DIR, "weights")
os.makedirs(weights_dir, exist_ok=True)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(weights_dir, "lrsnet_best.weights.h5"),
        save_best_only=True, save_weights_only=True, monitor="val_PSNR", mode="max",
    ),
    tf.keras.callbacks.CSVLogger(os.path.join(OUTPUT_DIR, "history_live.csv")),
]

## 8. Train

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
)

## 9. Save training figures, history, and final weights

In [ ]:
figures_dir = os.path.join(OUTPUT_DIR, "figures")

utils.plot_training_history(history, os.path.join(figures_dir, "lrsnet_loss_psnr.png"))
utils.save_history_csv(history, os.path.join(OUTPUT_DIR, "history.csv"))
model.save_weights(os.path.join(weights_dir, "lrsnet_final.weights.h5"))

## 10. Qualitative check

In [ ]:
lr_batch, hr_batch = next(iter(val_ds))
utils.visualize_predictions(
    model, lr_batch.numpy(), hr_batch.numpy(),
    os.path.join(figures_dir, "lrsnet_val_predictions.png"),
)

## 11. Efficiency report

In [ ]:
import json

report = utils.model_report(model)
print(report)
with open(os.path.join(OUTPUT_DIR, "model_report.json"), "w") as f:
    json.dump(report, f, indent=2)

## Bring back to CPU

Copy `OUTPUT_DIR` (weights, `history.csv`, `figures/`, `model_report.json`, and **`split.json`**) from Drive/Kaggle output into `Models/LRSNet/` locally, matching the existing `Models/SRCNN/`, `Models/EDSR/` layout. `split.json` records the exact train/val/test file lists — reuse its `"test"` entry (not a fresh random split) so the full-test-set comparison table, the params/PSNR Pareto plot against SRCNN/EDSR/SRGAN, and the downstream-task evaluation all run on the same held-out images.